In [3]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    precision_recall_curve, balanced_accuracy_score, matthews_corrcoef,
    confusion_matrix, f1_score
)

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Create the plots directory (if it doesn't exist)
os.makedirs("plots", exist_ok=True)

print("1) CLASS IMBALANCE — building a severely imbalanced dataset")


X, y = make_classification(
    n_samples=5000, n_features=15, n_informative=8, n_redundant=2,
    n_classes=2, weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
print(f"Class distribution: {np.bincount(y)}  (positive rate = {y.mean():.3f})")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train positive rate: {y_train.mean():.3f} | Test positive rate: {y_test.mean():.3f}")

print("\n--- Naive baseline: what does accuracy alone hide? ---")
naive_model = RandomForestClassifier(n_estimators=200, random_state=42)
naive_model.fit(X_train, y_train)
y_pred_naive = naive_model.predict(X_test)
print(f"Accuracy: {naive_model.score(X_test, y_test):.4f}  <- looks great, but...")
print(classification_report(y_test, y_pred_naive, digits=3))
print("Notice recall on the minority class (1) is much weaker than accuracy suggests.")


print("2) class_weight — the cheapest fix, no data resampling needed")


weighted_model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
weighted_model.fit(X_train, y_train)
y_pred_weighted = weighted_model.predict(X_test)
print(classification_report(y_test, y_pred_weighted, digits=3))
print(f"F1 (minority class) — naive: {f1_score(y_test, y_pred_naive):.4f} | "
      f"class_weight='balanced': {f1_score(y_test, y_pred_weighted):.4f}")


print("3) RANDOM OVERSAMPLING")


ros = RandomOverSampler(sampling_strategy=0.5, random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
print(f"Before oversampling: {np.bincount(y_train)}")
print(f"After oversampling : {np.bincount(y_ros)}")

model_ros = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_ros, y_ros)
print(classification_report(y_test, model_ros.predict(X_test), digits=3))


print("4) RANDOM UNDERSAMPLING")


rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
print(f"Before undersampling: {np.bincount(y_train)}")
print(f"After undersampling : {np.bincount(y_rus)}")

model_rus = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_rus, y_rus)
print(classification_report(y_test, model_rus.predict(X_test), digits=3))


print("5) SMOTE — synthetic minority oversampling")


smote = SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
print(f"Before SMOTE: {np.bincount(y_train)}")
print(f"After SMOTE : {np.bincount(y_smote)}")

model_smote = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_smote, y_smote)
print(classification_report(y_test, model_smote.predict(X_test), digits=3))

# --- Correct way: SMOTE inside a Pipeline, evaluated with cross-validation ---
print("\n--- SMOTE correctly wrapped in a Pipeline + StratifiedKFold CV ---")
print("(SMOTE is refit on each fold's TRAINING portion only — no leakage into validation folds)")
smote_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(smote_pipeline, X_train, y_train, cv=skf, scoring="average_precision")
print(f"CV PR-AUC (SMOTE pipeline): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")


print("6) PRECISION-RECALL CURVE — the right curve for imbalanced problems")


models = {
    "Naive (no fix)": naive_model,
    "class_weight=balanced": weighted_model,
    "Random Oversampling": model_ros,
    "Random Undersampling": model_rus,
    "SMOTE": model_smote,
}

plt.figure(figsize=(7, 6))
baseline_rate = y_test.mean()
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

plt.axhline(y=baseline_rate, linestyle="--", color="gray", label=f"No-skill baseline ({baseline_rate:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves — Comparing Imbalance-Handling Strategies")
plt.legend(fontsize=8)

# Save to the "plots" folder (created earlier)
plt.savefig("plots/24_pr_curves_comparison.png", bbox_inches="tight")
plt.close()
print("Saved comparison PR curve plot to plots/24_pr_curves_comparison.png")


print("7) APPROPRIATE EVALUATION METRICS — beyond accuracy")


print(f"{'Model':25s} {'Accuracy':>9s} {'F1(pos)':>9s} {'ROC-AUC':>9s} {'PR-AUC':>9s} {'BalAcc':>9s} {'MCC':>9s}")
for name, model in models.items():
    y_pred = model.predict(X_test)
    scores = model.predict_proba(X_test)[:, 1]
    acc = model.score(X_test, y_test)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, scores)
    pr = average_precision_score(y_test, scores)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    print(f"{name:25s} {acc:9.4f} {f1:9.4f} {roc:9.4f} {pr:9.4f} {bal_acc:9.4f} {mcc:9.4f}")

print("\nNote: accuracy stays high across the board (imbalance masks it), while F1/PR-AUC/")
print("balanced accuracy/MCC show much clearer differences between strategies —")
print("this is exactly why accuracy alone is the wrong metric for imbalanced problems.")

print("\nConfusion matrix — SMOTE model:")
print(confusion_matrix(y_test, model_smote.predict(X_test)))

print("\nDone.")

1) CLASS IMBALANCE — building a severely imbalanced dataset
Class distribution: [4727  273]  (positive rate = 0.055)
Train positive rate: 0.054 | Test positive rate: 0.055

--- Naive baseline: what does accuracy alone hide? ---
Accuracy: 0.9610  <- looks great, but...
              precision    recall  f1-score   support

           0      0.960     1.000     0.980       945
           1      1.000     0.291     0.451        55

    accuracy                          0.961      1000
   macro avg      0.980     0.645     0.715      1000
weighted avg      0.963     0.961     0.951      1000

Notice recall on the minority class (1) is much weaker than accuracy suggests.
2) class_weight — the cheapest fix, no data resampling needed
              precision    recall  f1-score   support

           0      0.957     1.000     0.978       945
           1      1.000     0.236     0.382        55

    accuracy                          0.958      1000
   macro avg      0.979     0.618     0.680  